# RRIN — Retinal Restoration Network Training on Kaggle

## HOW TO USE THIS NOTEBOOK (complete beginner guide)

This notebook fetches every retina image dataset directly through Kaggle's own API (using the `kagglehub` library). You never manually download, unzip, or attach image datasets through the website interface. The only thing uploaded by hand is your own project code, in Step 1 below.

### Step 1 — Upload the project code
1. On this Kaggle notebook page, click **File → Add Input** on the right sidebar
2. Click **Upload** → **New Dataset** → drag your project ZIP file
3. Name it `rrin-code` and publish it

(This is the only manual upload step. Your own code is not something `kagglehub` can fetch automatically — only the public image datasets are fetched through the API, in Cell 4 below.)

### Step 2 — Accept the competition rules (one time, in your browser)
`kagglehub` still requires that you have personally accepted each competition's rules before it is allowed to download files on your behalf. Open these two links, sign in, and click Accept:
- https://www.kaggle.com/c/diabetic-retinopathy-detection → click "Late Submission" → click "I Understand and Accept"
- https://www.kaggle.com/c/aptos2019-blindness-detection → click "Late Submission" → click "I Understand and Accept"

(RFMiD and ODIR-5K are open datasets, not competitions, so no acceptance step is needed for those two.)

### Step 3 — Add Hugging Face Secrets (optional — only for auto-upload)
You do **not** need to add your Kaggle username or API key as secrets. `kagglehub` authenticates automatically using your Kaggle login session when running inside a Kaggle notebook.

The only secrets you may want to add are for Hugging Face, so that CELL 9 can automatically upload your trained model when training finishes. If you skip this, training still completes normally — you just download `best.pt` manually from the Output panel afterward.

To add them:
1. In the **left sidebar** of this notebook editor, look for a **key icon** 🔑 or a section labelled **"Add-ons"** → **"Secrets"**, or a **padlock icon** 🔒
2. Click **"+ Add a new secret"**
3. Add `HUGGINGFACE_TOKEN` → value: your HF token starting with `hf_...`
4. Add `HF_MODEL_REPO` → value: `yourHFusername/rrin-retina-restoration`
5. Toggle both secrets **ON** in the same panel

### Step 4 (optional) — Add Hugging Face Secrets too
Only needed if you want the very last cell to automatically upload your trained model when training finishes. Add two more secrets the same way as Step 3:
- `HUGGINGFACE_TOKEN` → your Hugging Face write token
- `HF_MODEL_REPO` → e.g. `yourusername/rrin-retina-restoration`

If you skip this, training still completes normally — you just download `best.pt` manually from the output panel afterward instead.

### Step 5 — Enable the GPU
Click **Session options** (top right) → Accelerator → **GPU P100**

### Step 6 — Enable Internet access
Click **Session options** → confirm **Internet** is toggled ON (required for `kagglehub` to fetch files)

### Step 7 — Run all cells
Click **Run All**, or press Shift+Enter in each cell from top to bottom.

### Step 8 — Get your trained model
If you set up Step 4, it uploads automatically to Hugging Face. Otherwise, download `best.pt` from the output panel on the right.

---
**FREE GPU TIME:** Kaggle gives you 30 hours per week of free GPU compute. Training 200 epochs on a P100 GPU typically takes 12 to 20 hours.

**NO LOCAL STORAGE:** Every image dataset is fetched by `kagglehub` directly onto Kaggle's own servers. Nothing is ever downloaded to your own computer.

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# This installs the Python packages the project needs.
# Takes about 2-3 minutes. Run once.
# ============================================================
import subprocess, sys

# kagglehub needs to be reasonably recent for competition_download() to exist
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kagglehub>=0.3.4'])

packages = [
    'noise==1.2.2',
    'albumentations>=1.3.0',
    'lpips>=0.1.4',
    'fastapi>=0.104.0',
    'uvicorn[standard]>=0.24.0',
    'python-multipart>=0.0.6',
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages installed.')

In [ ]:
# ============================================================
# CELL 2 — Copy project code and set up paths
# ============================================================
import os, shutil, sys

# Copy project code from the input dataset to the working directory
PROJECT_CODE_PATH = '/kaggle/input/rrin-code'
WORK_DIR = '/kaggle/working/rrin_project'

if os.path.exists(PROJECT_CODE_PATH):
    shutil.copytree(PROJECT_CODE_PATH, WORK_DIR, dirs_exist_ok=True)
    print(f'Code copied to {WORK_DIR}')
else:
    print('ERROR: Upload your project code as a Kaggle dataset named rrin-code')
    print('See the instructions in the first cell above.')

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print(f'Working directory: {os.getcwd()}')

# Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found! Enable GPU in Session options -> Accelerator -> GPU P100')

In [ ]:
# ============================================================
# CELL 3 — Load Hugging Face credentials from Kaggle Secrets
#
# KAGGLE credentials are NOT needed here. When this notebook
# runs inside Kaggle, kagglehub automatically authenticates
# using your logged-in Kaggle session — no username or key
# needs to be typed anywhere.
#
# The only secrets needed are the optional Hugging Face ones,
# which let CELL 9 auto-upload your trained model at the end.
# If you skipped adding those secrets, this cell prints a short
# notice and continues — training is not affected.
# ============================================================
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()

def try_load_secret(name):
    """Load one secret. Returns the value, or None if not found."""
    try:
        value = secrets.get_secret(name)
        os.environ[name] = value
        print('  ' + name + ': loaded OK')
        return value
    except Exception:
        print('  ' + name + ': not set (optional — see notebook instructions cell)')
        return None

print('Loading optional Hugging Face credentials...')
try_load_secret('HUGGINGFACE_TOKEN')
try_load_secret('HF_MODEL_REPO')
print()
print('Note: KAGGLE_USERNAME and KAGGLE_KEY are not needed here.')
print('      kagglehub uses your Kaggle login session automatically.')


In [ ]:
# ============================================================
# CELL 4 — Fetch every dataset through the Kaggle API
# (kagglehub) and build config_kaggle.yaml automatically from
# the real paths kagglehub returns. You never type a folder
# path yourself, and you never need to know the internal
# folder structure inside any of these datasets.
# ============================================================
import kagglehub
import os

print('Fetching datasets through the Kaggle API.')
print('These are served directly from Kaggle storage and do not use your own computer at all.\n')

# ---- "Datasets" use kagglehub.dataset_download ----
print('RFMiD (Retinal Disease Classification)...')
rfmid_path = kagglehub.dataset_download('andrewmvd/retinal-disease-classification')
print('  -> ' + rfmid_path)

print('ODIR-5K (Ocular Disease Recognition)...')
odir_path = kagglehub.dataset_download('andrewmvd/ocular-disease-recognition-odir5k')
print('  -> ' + odir_path)

# ---- "Competitions" use kagglehub.competition_download ----
# This requires that you accepted each competition's rules in your
# browser already (see the instructions cell at the top of this notebook).
print('APTOS 2019 Blindness Detection (competition)...')
aptos_path = kagglehub.competition_download('aptos2019-blindness-detection')
print('  -> ' + aptos_path)

print('Diabetic Retinopathy Detection / EyePACS (competition)...')
eyepacs_path = kagglehub.competition_download('diabetic-retinopathy-detection')
print('  -> ' + eyepacs_path)

print('\nAll four datasets fetched. Building config_kaggle.yaml...')

# NOTE: each path above is the TOP-LEVEL folder kagglehub returned.
# dataset_paths is intentionally pointed at this top level rather than
# any specific inner subfolder. The ingestion step searches every
# subfolder automatically and recursively (see src/database.py), so
# every image is found regardless of how the zip file was organised
# internally. You do not need to know or choose a subfolder name.

KAGGLE_CONFIG = f'''
dataset_paths:
  eyepacs:   "{eyepacs_path}"
  aptos:     "{aptos_path}"
  rfmid:     "{rfmid_path}"
  idrid:     ""   # Not available on Kaggle -- leave blank
  messidor2: ""   # Not available on Kaggle -- leave blank
  odir:      "{odir_path}"
  stare:     ""
  drive:     ""

metadata_db_path:  "/kaggle/working/metadata/retina_restoration_metadata.sqlite"
checkpoint_dir:    "/kaggle/working/checkpoints/"
log_dir:           "/kaggle/working/logs/"

image_size:      256
crop_size:       256
input_channels:  4
output_channels: 3

base_filters:           64
num_residual_blocks:    6
attention_gate_enabled: true

batch_size:          4
num_workers:         2
num_epochs:          200
lr_constant_epochs:  100
lr_decay_epochs:     100

learning_rate:  0.0002
adam_beta1:     0.5
adam_beta2:     0.999
adam_epsilon:   1.0e-8
weight_decay:   1.0e-5

lambda_adv:        1.0
lambda_l1:         100.0
lambda_ssim:       10.0
lambda_perceptual: 10.0
lambda_cycle:      10.0

label_smoothing_real_target: 0.9
early_stopping_patience:     15

vgg_perceptual_layers:
  - "relu2_2"
  - "relu3_3"

random_seed: 42
train_fraction:             0.85
val_fraction:               0.10
quality_quantile_threshold: 0.75

blur_probability:         0.6
num_reflections_range:    [1, 4]
jpeg_quality_range:       [35, 80]
'''

os.environ['CONFIG_PATH'] = '/kaggle/working/rrin_project/config_kaggle.yaml'
with open('/kaggle/working/rrin_project/config_kaggle.yaml', 'w') as f:
    f.write(KAGGLE_CONFIG)

for d in ['/kaggle/working/metadata', '/kaggle/working/checkpoints', '/kaggle/working/logs']:
    os.makedirs(d, exist_ok=True)

print('config_kaggle.yaml written automatically using the paths fetched above.')
print('Output directories created.')

In [ ]:
# ============================================================
# CELL 5 — Ingest datasets and run quality scoring
# This reads every image file path into the SQLite database.
# Then computes quality scores to find the best training images.
# ============================================================
import sqlite3
from src.config import DATASET_PATHS, METADATA_DB_PATH
from src.database import (
    initialize_metadata_database,
    ingest_source_dataset_into_database,
    load_metadata_as_dataframe,
    build_image_id_to_split_mapping,
    write_split_assignments_to_database,
)
from src.quality_scoring import compute_and_attach_all_quality_scores, select_pseudo_clean_pool
from src.splits import assign_patient_grouped_splits, verify_no_patient_overlap_across_splits

connection, cursor = initialize_metadata_database(METADATA_DB_PATH)

for dataset_name, dataset_path in DATASET_PATHS.items():
    if dataset_path and os.path.isdir(str(dataset_path)):
        n = ingest_source_dataset_into_database(connection, cursor, str(dataset_path), dataset_name)
        print(f'Ingested {n} images from {dataset_name}')
    else:
        if dataset_path:
            print(f'Skipping {dataset_name} — path not found: {dataset_path}')

df = load_metadata_as_dataframe(connection)
print(f'\nTotal images registered: {len(df)}')

print('\nComputing quality scores (this takes 20-60 minutes for large datasets)...')
df = compute_and_attach_all_quality_scores(df, connection, cursor)
df = select_pseudo_clean_pool(df)

df = assign_patient_grouped_splits(df)
verify_no_patient_overlap_across_splits(df)
write_split_assignments_to_database(connection, cursor, build_image_id_to_split_mapping(df))

print('\nDataset preparation complete!')
print(df['split_assignment'].value_counts())

In [ ]:
# ============================================================
# CELL 6 — Build models and start training
# This is the main training loop.
# Expected time: 12-20 hours on a Kaggle P100 GPU
# ============================================================
import torch
import torch.utils.data

from src.config import DEVICE, LEARNING_RATE, ADAM_BETA1, ADAM_BETA2, ADAM_EPSILON, WEIGHT_DECAY, NUM_EPOCHS, BATCH_SIZE, NUM_WORKERS, CHECKPOINT_DIR, LOG_DIR, RANDOM_SEED
from src.datasets import RetinaRestorationDataset
from src.models.generator import UNetGenerator, initialize_network_weights
from src.models.discriminator import PatchGANDiscriminator
from src.models.losses import VGGPerceptualLoss, SSIMLoss
from src.training.train import train_one_epoch, validate_one_epoch
from src.training.checkpoints import save_checkpoint, EarlyStopping, get_linear_decay_lr_scheduler
from src.utils.logging_utils import setup_logger, set_global_random_seeds

set_global_random_seeds(RANDOM_SEED)
logger = setup_logger(LOG_DIR)

train_ds = RetinaRestorationDataset(df, 'train', is_training=True)
val_ds   = RetinaRestorationDataset(df, 'val',   is_training=False)

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
val_dl   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

generator     = UNetGenerator().to(DEVICE)
discriminator = PatchGANDiscriminator().to(DEVICE)
generator.apply(initialize_network_weights)
discriminator.apply(initialize_network_weights)

gen_opt  = torch.optim.Adam(generator.parameters(),     lr=LEARNING_RATE, betas=(ADAM_BETA1, ADAM_BETA2), eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
disc_opt = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(ADAM_BETA1, ADAM_BETA2), eps=ADAM_EPSILON)
gen_sched  = get_linear_decay_lr_scheduler(gen_opt)
disc_sched = get_linear_decay_lr_scheduler(disc_opt)

perc_loss  = VGGPerceptualLoss().to(DEVICE)
ssim_loss  = SSIMLoss().to(DEVICE)
early_stop = EarlyStopping(metric_should_increase=True)

print(f'Training {len(train_ds)} images for {NUM_EPOCHS} epochs on {DEVICE}')
print('Best checkpoint will be saved to:', CHECKPOINT_DIR + 'best.pt')

for epoch in range(NUM_EPOCHS):
    train_one_epoch(generator, discriminator, gen_opt, disc_opt, perc_loss, ssim_loss, train_dl, epoch, logger)
    val_metrics = validate_one_epoch(generator, val_dl, epoch, logger)
    gen_sched.step()
    disc_sched.step()
    should_stop, is_best = early_stop.step(val_metrics['ssim'])
    save_checkpoint(CHECKPOINT_DIR, epoch, generator, discriminator, gen_opt, disc_opt, val_metrics, is_best)
    if should_stop:
        print(f'Early stopping at epoch {epoch}'); break

print('Training complete! Download best.pt from the output panel on the right.')

In [ ]:
# ============================================================
# CELL 7 — Run evaluation on the held-out test set
# ============================================================
from src.datasets import RetinaRestorationDataset
from src.training.checkpoints import load_checkpoint
from src.evaluation.metrics import evaluate_on_test_set

test_ds = RetinaRestorationDataset(df, 'test', is_training=False)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)

# Load the best checkpoint
best_path = '/kaggle/working/checkpoints/best.pt'
if os.path.exists(best_path):
    load_checkpoint(best_path, generator, discriminator)
    results_df = evaluate_on_test_set(generator, test_dl, None, None, logger)
    results_df.to_csv('/kaggle/working/test_results.csv', index=False)
    print('\nEvaluation results saved to /kaggle/working/test_results.csv')
else:
    print('No best checkpoint found. Run training first (Cell 5).')

In [ ]:
# ============================================================
# CELL 8 — Visualise a sample restoration
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

from src.inference.restore import restore_image_array, load_generator_for_inference
from src.utils.image_utils import load_image_as_float_array

# Pick a random test image to visualise
test_rows = df[df['split_assignment'] == 'test'].head(3)

gen_inf = load_generator_for_inference('/kaggle/working/checkpoints/best.pt')

fig, axes = plt.subplots(3, 2, figsize=(12, 15))
from src.degradation import compose_random_degradation_pipeline
from src.quality_scoring import extract_field_of_view_mask

for i, (_, row) in enumerate(test_rows.iterrows()):
    clean   = load_image_as_float_array(row['file_path'])
    uint8_img = (clean * 255).astype(np.uint8)
    fov_mask  = extract_field_of_view_mask(uint8_img)
    np.random.seed(i)
    degraded  = compose_random_degradation_pipeline(clean, fov_mask)
    restored  = restore_image_array(gen_inf, degraded)
    axes[i, 0].imshow(degraded);  axes[i, 0].set_title('Degraded Input'); axes[i, 0].axis('off')
    axes[i, 1].imshow(restored);  axes[i, 1].set_title('Restored Output'); axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_restorations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample saved to /kaggle/working/sample_restorations.png')

In [ ]:
# ============================================================
# CELL 9 — Upload the trained model to Hugging Face (optional)
# Skips automatically if the HUGGINGFACE_TOKEN / HF_MODEL_REPO
# secrets were not set up in the instructions cell at the top.
# ============================================================
import os, subprocess, sys

token = os.environ.get('HUGGINGFACE_TOKEN')
repo  = os.environ.get('HF_MODEL_REPO')

if token and repo:
    subprocess.check_call([
        sys.executable,
        '/kaggle/working/rrin_project/scripts/upload_model_to_huggingface.py',
        '--token', token,
        '--repo',  repo,
        '--checkpoint', '/kaggle/working/checkpoints/best.pt',
    ])
    print('Model uploaded to Hugging Face.')
else:
    print('HUGGINGFACE_TOKEN / HF_MODEL_REPO secrets were not set, so the upload was skipped.')
    print('Your trained model is still saved at /kaggle/working/checkpoints/best.pt')
    print('Download it manually from the Output panel on the right, or add the two')
    print('secrets described in Step 4 of the instructions cell and re-run this cell.')